In [1]:
import pandas as pd
import os
from datetime import datetime, timedelta

In [6]:
directory = 'C:/Users/tuan-/Downloads/1 Lobster Thesis/Data/SPY2016/'

# Konverter tid
def convert_to_datetime(seconds, base_date):
    base_time = datetime.strptime(base_date, '%Y-%m-%d')
    return base_time + timedelta(seconds=seconds)


monthly_data = []


year = 2016

for month in range(1, 13):  # Loop monthly 
    monthly_files = []
    
    for filename in os.listdir(directory):
        if f'{year}-{month:02}' in filename and 'message' in filename:  # Match filer monthly and year
            message_file = os.path.join(directory, filename)
            orderbook_file = message_file.replace('message', 'orderbook')  # Match orderbook fil
            
            # Load message and orderbook 
            message_df = pd.read_csv(message_file)
            orderbook_df = pd.read_csv(orderbook_file)

            # Message and orderbook data, dropper col 7
            message_df = message_df.iloc[:, :-1]  # 
            message_df.columns = ['Time (sec)', 'Event Type', 'Order ID', 'Size', 'Price', 'Direction']
            orderbook_df.columns = ['Ask Price 1', 'Ask Size 1', 'Bid Price 1', 'Bid Size 1', 
                                    'Ask Price 2', 'Ask Size 2', 'Bid Price 2', 'Bid Size 2']

            base_date = filename.split('_')[1]  # Start dag SPY_2016-01-04
            message_df['Time (sec)'] = message_df['Time (sec)'].apply(lambda x: convert_to_datetime(x, base_date))

            # Merger message and orderbook data on 'Time (sec)'
            combined_df = pd.merge(orderbook_df, message_df, left_index=True, right_index=True, how='left')

            # Drop NAN
            combined_df.dropna(inplace=True)

            # Set 'Time (sec)' index for 1 second
            combined_df.set_index('Time (sec)', inplace=True)

            # Sampler data 1 sec interval første obs hvert sec
            resampled_df = combined_df.resample('1S').first()

            # list monthly
            monthly_files.append(resampled_df)
    
    # Concatenate all daily filer for month
    if monthly_files:
        monthly_data_df = pd.concat(monthly_files)
        monthly_data.append(monthly_data_df)
        
        # Gem
        monthly_data_df.to_csv(f'processed_{year}_{month:02}.csv', index=False)

# All months into one final DataFrame
final_df = pd.concat(monthly_data)

# Save the final DataFrame, year 2016
final_df.to_csv(f'combined_SPY{year}_cleaned.csv', index=False)


print(final_df.head())

In [16]:
print(final_df.tail())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2016-12-30 15:59:55    2234600.0      5200.0    2234500.0      3900.0   
2016-12-30 15:59:56    2234400.0      4500.0    2234200.0     12300.0   
2016-12-30 15:59:57    2234700.0      5586.0    2234500.0     17300.0   
2016-12-30 15:59:58    2234800.0      4100.0    2234700.0      6000.0   
2016-12-30 15:59:59    2235000.0     18000.0    2234800.0     14800.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2016-12-30 15:59:55    2234700.0     10800.0    2234400.0     17800.0   
2016-12-30 15:59:56    2234500.0      6800.0    2234100.0      7000.0   
2016-12-30 15:59:57    2234800.0     13100.0    2234400.0     11200.0   
2016-12-30 15:59:58    2234900.0     13200.0    2234600.0     10400.0   
2016-12-30 15:59:59    2235100.0     15500.0    22

In [7]:
print(final_df.head())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2016-01-04 09:30:00    2005100.0      4200.0    2004900.0       100.0   
2016-01-04 09:30:01    2003200.0       100.0    2003100.0       200.0   
2016-01-04 09:30:02    2003500.0       433.0    2003400.0       200.0   
2016-01-04 09:30:03    2003800.0       900.0    2003700.0       200.0   
2016-01-04 09:30:04    2003700.0       800.0    2003500.0      2600.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2016-01-04 09:30:00    2005200.0       900.0    2004800.0      4400.0   
2016-01-04 09:30:01    2003300.0      6600.0    2003000.0       191.0   
2016-01-04 09:30:02    2003600.0      1400.0    2003300.0       900.0   
2016-01-04 09:30:03    2003900.0      4300.0    2003600.0       100.0   
2016-01-04 09:30:04    2003800.0      7900.0    20

In [8]:
# Observations (rows) in the final combined DataFrame for the year
total_observations_final = len(final_df)

print(f"Total observations in the final combined DataFrame: {total_observations_final}")

Total observations in the final combined DataFrame: 5896761


In [36]:
final_df.head()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction,Date
Time (sec),,,,,,,,,,,,,,
2016-01-04 09:30:00,2005100.0,4200.0,2004900.0,100.0,2005200.0,900.0,2004800.0,4400.0,1.0,6807043.0,100.0,2004900.0,1.0,2016-01-04
2016-01-04 09:30:01,2003200.0,100.0,2003100.0,200.0,2003300.0,6600.0,2003000.0,191.0,1.0,7095697.0,6000.0,2003300.0,-1.0,2016-01-04
2016-01-04 09:30:02,2003500.0,433.0,2003400.0,200.0,2003600.0,1400.0,2003300.0,900.0,4.0,7272137.0,67.0,2003500.0,-1.0,2016-01-04
2016-01-04 09:30:03,2003800.0,900.0,2003700.0,200.0,2003900.0,4300.0,2003600.0,100.0,1.0,7493033.0,100.0,2003800.0,-1.0,2016-01-04
2016-01-04 09:30:04,2003700.0,800.0,2003500.0,2600.0,2003800.0,7900.0,2003400.0,1400.0,1.0,7564191.0,1500.0,2003500.0,1.0,2016-01-04


In [37]:
final_df.tail()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction,Date
Time (sec),,,,,,,,,,,,,,
2016-12-30 15:59:55,2234600.0,5200.0,2234500.0,3900.0,2234700.0,10800.0,2234400.0,17800.0,3.0,198619488.0,800.0,2234600.0,-1.0,2016-12-30
2016-12-30 15:59:56,2234400.0,4500.0,2234200.0,12300.0,2234500.0,6800.0,2234100.0,7000.0,3.0,198692608.0,800.0,2234200.0,1.0,2016-12-30
2016-12-30 15:59:57,2234700.0,5586.0,2234500.0,17300.0,2234800.0,13100.0,2234400.0,11200.0,3.0,198745056.0,2000.0,2234500.0,1.0,2016-12-30
2016-12-30 15:59:58,2234800.0,4100.0,2234700.0,6000.0,2234900.0,13200.0,2234600.0,10400.0,1.0,198815352.0,2000.0,2234800.0,-1.0,2016-12-30
2016-12-30 15:59:59,2235000.0,18000.0,2234800.0,14800.0,2235100.0,15500.0,2234700.0,26300.0,1.0,198879460.0,500.0,2234800.0,1.0,2016-12-30


In [82]:
# Save 
final_df.to_csv('final_combined_2016.csv', index=True)  # Keep the index to retain 'Time (sec)'

final_df.to_csv('final_combined_2016_with_time.csv', index=True)

In [35]:
# Count unique days
final_df['Date'] = final_df.index.date

# Count the number of unique trading days
unique_trading_days = final_df['Date'].nunique()

print(f"Number of unique trading days: {unique_trading_days}")

Number of unique trading days: 252


In [77]:
# Load the data from 'final_combined_2016_with_time.csv'
final_combined_2016_with_time = pd.read_csv('final_combined_2016_with_time.csv')


final_combined_2016_with_time['Time (sec)'] = pd.to_datetime(final_combined_2016_with_time['Time (sec)'])

# Group trading days
grouped = final_combined_2016_with_time.groupby(final_combined_2016_with_time['Time (sec)'].dt.date)

# Sampler liste
resampled_5min_list = []

# Iterate through each group each day
for date, group in grouped:
    # Filtrer data for trading hours (9:30 AM to 4:00 PM)
    group_trading_hours = group[
        (group['Time (sec)'].dt.time >= pd.to_datetime('09:30:00').time()) &
        (group['Time (sec)'].dt.time <= pd.to_datetime('16:00:00').time())
    ]
    
    # Resample for 5-minute intervals 
    resampled_day = group_trading_hours.resample('5T', on='Time (sec)').agg({
        'Ask Price 1': ['first', 'max', 'min', 'last'],
        'Bid Price 1': ['first', 'max', 'min', 'last'],
        'Ask Price 2': ['first', 'max', 'min', 'last'],
        'Bid Price 2': ['first', 'max', 'min', 'last'],
        'Ask Size 1': ['sum', 'mean'],
        'Bid Size 1': ['sum', 'mean'],
        'Ask Size 2': ['sum', 'mean'],
        'Bid Size 2': ['sum', 'mean'],
        'Price': ['first', 'max', 'min', 'last'],
        'Direction': 'mean'
    })
    
   
    resampled_day.columns = ['_'.join(col).strip() for col in resampled_day.columns.values]
    
    # Append the resampled data dag
    resampled_5min_list.append(resampled_day)

# Concatenate all, DataFrame
resampled_5min_final_df = pd.concat(resampled_5min_list)

# Reset index 'Time (sec)' 
resampled_5min_final_df.reset_index(inplace=True)

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2016-01-04 09:30:00          2005100.0        2005100.0        2000400.0   
1 2016-01-04 09:35:00          2000800.0        2004200.0        2000800.0   
2 2016-01-04 09:40:00          2002500.0        2002500.0        1996700.0   
3 2016-01-04 09:45:00          1997700.0        2001000.0        1996100.0   
4 2016-01-04 09:50:00          2000500.0        2004800.0        2000500.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         2000900.0          2004900.0        2004900.0        2000200.0   
1         2002500.0          2000700.0        2004100.0        2000700.0   
2         1997500.0          2002400.0        2002400.0        1996600.0   
3         2000500.0          1997600.0        2000800.0        1996000.0   
4         2003000.0          2000400.0        2004700.0        2000400.0   

   Bid Price 1_last  Ask Price 2_first  ...  Bid Size 1_mean  Ask Size 2_s

In [78]:
print(resampled_5min_final_df.head(10))

# Save
resampled_5min_final_df.to_csv('resampled_5min_final_2016_corrected.csv', index=False)

            Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0  2016-01-04 09:30:00          2005100.0        2005100.0        2000400.0   
1  2016-01-04 09:35:00          2000800.0        2004200.0        2000800.0   
2  2016-01-04 09:40:00          2002500.0        2002500.0        1996700.0   
3  2016-01-04 09:45:00          1997700.0        2001000.0        1996100.0   
4  2016-01-04 09:50:00          2000500.0        2004800.0        2000500.0   
..                 ...                ...              ...              ...   
74 2016-01-04 15:40:00          1996900.0        2000800.0        1996800.0   
75 2016-01-04 15:45:00          2000300.0        2006800.0        2000300.0   
76 2016-01-04 15:50:00          2006300.0        2007300.0        2003700.0   
77 2016-01-04 15:55:00          2005900.0        2010200.0        2005700.0   
78 2016-01-05 09:30:00          2014200.0        2018100.0        2012400.0   

    Ask Price 1_last  Bid Price 1_first  Bid Price 

In [79]:
# Count observations for each day in 5-minute interval
resampled_5min_final_df['Date'] = resampled_5min_final_df['Time (sec)'].dt.date

# Count the number of rows for each day
observations_per_day = resampled_5min_final_df.groupby('Date').size()


print(observations_per_day)

Date
2016-01-04    78
2016-01-05    78
2016-01-06    78
2016-01-07    78
2016-01-08    78
              ..
2016-12-23    78
2016-12-27    78
2016-12-28    78
2016-12-29    78
2016-12-30    78
Length: 252, dtype: int64


In [80]:
print(resampled_5min_final_df.head(10))

# Save igen
resampled_5min_final_df.to_csv('resampled_5min_final_2016_corrected.csv', index=False)

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2016-01-04 09:30:00          2005100.0        2005100.0        2000400.0   
1 2016-01-04 09:35:00          2000800.0        2004200.0        2000800.0   
2 2016-01-04 09:40:00          2002500.0        2002500.0        1996700.0   
3 2016-01-04 09:45:00          1997700.0        2001000.0        1996100.0   
4 2016-01-04 09:50:00          2000500.0        2004800.0        2000500.0   
5 2016-01-04 09:55:00          2003000.0        2004000.0        2002000.0   
6 2016-01-04 10:00:00          2002400.0        2002400.0        1994300.0   
7 2016-01-04 10:05:00          1995000.0        1998500.0        1994300.0   
8 2016-01-04 10:10:00          1998200.0        2000100.0        1996200.0   
9 2016-01-04 10:15:00          1999900.0        2001300.0        1999600.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         2000900.0          2004900.0        2004900.0        20

In [81]:
from IPython.display import display, HTML

# Display as an HTML tabel
html_table = resampled_5min_final_df.head(10).to_html(index=False)
display(HTML(html_table))

# Gem denne
resampled_5min_final_df.to_csv('resampled_5min_final_2016_corrected.csv', index=False)


Time (sec),Ask Price 1_first,Ask Price 1_max,Ask Price 1_min,Ask Price 1_last,Bid Price 1_first,Bid Price 1_max,Bid Price 1_min,Bid Price 1_last,Ask Price 2_first,Ask Price 2_max,Ask Price 2_min,Ask Price 2_last,Bid Price 2_first,Bid Price 2_max,Bid Price 2_min,Bid Price 2_last,Ask Size 1_sum,Ask Size 1_mean,Bid Size 1_sum,Bid Size 1_mean,Ask Size 2_sum,Ask Size 2_mean,Bid Size 2_sum,Bid Size 2_mean,Price_first,Price_max,Price_min,Price_last,Direction_mean,Date
2016-01-04 09:30:00,2005100.0,2005100.0,2000400.0,2000900.0,2004900.0,2004900.0,2000200.0,2000800.0,2005200.0,2005200.0,2000500.0,2001000.0,2004800.0,2004800.0,2000100.0,2000700.0,579621.0,1932.070000,364161.0,1213.870000,943513.0,3145.043333,604276.0,2014.253333,2004900.0,2004900.0,2000300.0,2001000.0,0.073333,2016-01-04
2016-01-04 09:35:00,2000800.0,2004200.0,2000800.0,2002500.0,2000700.0,2004100.0,2000700.0,2002400.0,2000900.0,2004300.0,2000900.0,2002600.0,2000600.0,2004000.0,2000600.0,2002300.0,318026.0,1060.086667,260809.0,869.363333,603585.0,2011.950000,401070.0,1336.900000,2000700.0,2004100.0,2000700.0,2002300.0,0.026667,2016-01-04
2016-01-04 09:40:00,2002500.0,2002500.0,1996700.0,1997500.0,2002400.0,2002400.0,1996600.0,1997400.0,2002600.0,2002600.0,1996800.0,1997600.0,2002300.0,2002300.0,1996500.0,1997300.0,322954.0,1076.513333,260321.0,867.736667,636692.0,2122.306667,475493.0,1584.976667,2002600.0,2002600.0,1996500.0,1997300.0,0.106667,2016-01-04
2016-01-04 09:45:00,1997700.0,2001000.0,1996100.0,2000500.0,1997600.0,2000800.0,1996000.0,2000400.0,1997800.0,2001100.0,1996200.0,2000600.0,1997500.0,2000700.0,1995900.0,2000300.0,355166.0,1183.886667,300447.0,1001.490000,673242.0,2244.140000,481944.0,1606.480000,1997600.0,2001000.0,1996000.0,2000400.0,-0.133333,2016-01-04
2016-01-04 09:50:00,2000500.0,2004800.0,2000500.0,2003000.0,2000400.0,2004700.0,2000400.0,2002900.0,2000600.0,2004900.0,2000600.0,2003100.0,2000300.0,2004600.0,2000300.0,2002800.0,290215.0,967.383333,289818.0,966.060000,519021.0,1730.070000,626056.0,2086.853333,2000500.0,2004800.0,2000400.0,2003000.0,-0.120000,2016-01-04
2016-01-04 09:55:00,2003000.0,2004000.0,2002000.0,2002400.0,2002900.0,2003800.0,2001900.0,2002200.0,2003100.0,2004100.0,2002100.0,2002500.0,2002800.0,2003700.0,2001800.0,2002100.0,211902.0,713.474747,218519.0,735.754209,453695.0,1527.592593,561208.0,1889.589226,2002900.0,2004000.0,2002000.0,2002200.0,0.050505,2016-01-04
2016-01-04 10:00:00,2002400.0,2002400.0,1994300.0,1995200.0,2002000.0,2002000.0,1994200.0,1995100.0,2002500.0,2002500.0,1994400.0,1995300.0,2001900.0,2001900.0,1994100.0,1995000.0,310518.0,1035.060000,615111.0,2050.370000,595257.0,1984.190000,937914.0,3126.380000,2002100.0,2002100.0,1994300.0,1995000.0,-0.046667,2016-01-04
2016-01-04 10:05:00,1995000.0,1998500.0,1994300.0,1997800.0,1994800.0,1998300.0,1994100.0,1997700.0,1995100.0,1998600.0,1994400.0,1997900.0,1994700.0,1998200.0,1994000.0,1997600.0,311938.0,1039.793333,278395.0,927.983333,750038.0,2500.126667,655827.0,2186.090000,1995000.0,1998500.0,1994200.0,1997700.0,0.093333,2016-01-04
2016-01-04 10:10:00,1998200.0,2000100.0,1996200.0,2000100.0,1998100.0,2000000.0,1996100.0,2000000.0,1998300.0,2000200.0,1996300.0,2000200.0,1998000.0,1999900.0,1996000.0,1999900.0,287219.0,957.396667,251829.0,839.430000,670531.0,2235.103333,651349.0,2171.163333,1998300.0,2000100.0,1996100.0,1999900.0,-0.053333,2016-01-04
2016-01-04 10:15:00,1999900.0,2001300.0,1999600.0,2001100.0,1999700.0,2001200.0,1999500.0,2001000.0,2000000.0,2001400.0,1999700.0,2001200.0,1999600.0,2001100.0,1999400.0,2000900.0,354386.0,1181.286667,278241.0,927.470000,703914.0,2346.380000,752015.0,2506.716667,1999800.0,2001400.0,1999500.0,2001000.0,0.000000,2016-01-04
